# Phase 4 - Causal Activation Steering

Este notebook analisa se uma direcao latente extraida na Fase 3 causa mudanca mensuravel na geracao. O foco e comparar direcao correta, direcoes negativas e controles aleatorios.

A primeira celula configura caminhos e importa bibliotecas. Ela nao assume que os experimentos ja foram executados.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
PHASE4_DIR = ROOT / 'runs' / 'phase4'
PHASE4_DIR

Aqui carregamos todos os arquivos `*_summary.json` da Fase 4 e extraimos apenas as colunas necessarias para comparacao experimental.

In [ ]:
rows = []
for path in sorted(PHASE4_DIR.rglob('*_summary.json')) if PHASE4_DIR.exists() else []:
    data = json.loads(path.read_text(encoding='utf-8'))
    steering = data.get('steering', {})
    rows.append({
        'run': path.stem.replace('_summary', ''),
        'layer': steering.get('layer'),
        'alpha': steering.get('alpha'),
        'direction_type': steering.get('direction_type'),
        'accuracy': data.get('observed_best_of_n'),
        'pass_at_1': data.get('strict_pass_at_1'),
        'tokens_to_success': data.get('mean_tokens_until_success_or_budget'),
        'diversity': data.get('diversity_score'),
        'collapse': data.get('collapse_score'),
    })

phase4 = pd.DataFrame(rows)
phase4

Esta visualizacao mostra se o alpha positivo desloca a acuracia em relacao aos controles. Se a tabela estiver vazia, rode o sweep indicado no README.

In [ ]:
if phase4.empty:
    print('Sem resultados da Fase 4 ainda.')
else:
    ax = phase4.pivot_table(index='alpha', columns='direction_type', values='accuracy', aggfunc='mean').plot(marker='o')
    ax.set_ylabel('Best-of-N observado')
    ax.set_title('Efeito causal por alpha e controle')
    plt.show()

A proxima celula destaca candidatos promissores: maior acuracia com custo menor ou semelhante. Use isto como evidencia para escolher camadas em steering futuro.

In [ ]:
if phase4.empty:
    candidates = pd.DataFrame()
else:
    candidates = phase4.sort_values(['accuracy', 'tokens_to_success'], ascending=[False, True]).head(10)
candidates

Comando minimo para gerar dados desta fase:

`python scripts/run_steering_sweep.py --limit 5 --n 2 --alphas 0,1 --layers 12 --skip-existing`